# NB16 — Figuras de fiabilidad y estructura espectral para publicación

**Preguntas:** ¿coincide la confianza del clasificador con su exactitud? ¿qué mejora o empeora después de calibrar? ¿cómo se relacionan la estructura espectral, los dominios y las etapas?

Solo CPU. Regenera figuras usando las tablas congeladas NB06-v1.1 y las coordenadas NB15. **No entrena modelos, no ajusta calibración y no vuelve a ajustar PCA/UMAP.** Ejes de fiabilidad 0–1 idénticos; 15 intervalos originales; vacíos omitidos; brecha respecto a confianza media real. Los histogramas muestran cuántas observaciones sostienen cada barra. No se asignan datos a intervalos vacíos.

Las figuras de fiabilidad corresponden al análisis RAW original (17.662 observaciones repetidas en cuatro tests); las proyecciones NB15 corresponden a 1.795 asociaciones firma–etapa, N=25. Son poblaciones diferentes, y no se mezclan. Las figuras suplementarias identifican como diagnósticas las probabilidades derivadas de puntuaciones SVM/PLSDA. Solo los modelos espectrales seleccionados con probabilidades nativas forman la figura principal.

Exportación PNG 600 dpi, PDF y SVG, tablas, notas de interpretación, trazabilidad y ZIP descargable. Cada representación PCA/UMAP tiene su propio sistema de ejes; el acercamiento FULL_TARGET puede ser inducido por el centrado por dominio y no prueba transferencia predictiva.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime,timezone
import io,json,hashlib,zipfile,shutil,subprocess,sys
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap,BoundaryNorm
PROJECT=Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
RESULTS=PROJECT/'06_RESULTS'
NB06_OVERRIDE=None
NB15_ZIP_OVERRIDE=None
candidates=sorted([p for p in RESULTS.glob('NB06_V1_1_WIDE_TEMPERATURE_*') if p.is_dir()])
NB06=Path(NB06_OVERRIDE) if NB06_OVERRIDE else (candidates[-1] if candidates else None)
if NB06 is None:raise FileNotFoundError('NB06-v1.1 folder not found in 06_RESULTS')
zs=sorted(RESULTS.glob('RESULTS_NB15_SPECTRAL_ERROR_AUDIT_*.zip'),key=lambda p:p.stat().st_mtime,reverse=True)
NB15=Path(NB15_ZIP_OVERRIDE) if NB15_ZIP_OVERRIDE else (zs[0] if zs else None)
if NB15 is None:raise FileNotFoundError('Completed NB15 ZIP not found in 06_RESULTS')
RUN=RESULTS/('NB16_PUBLICATION_FIGURES_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
FIG=RUN/'figures';TABLE=RUN/'tables';META=RUN/'metadata'
for p in [FIG,TABLE,META]:p.mkdir(parents=True,exist_ok=False)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.labelsize':11,'axes.titlesize':11,'axes.titleweight':'semibold','axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'svg.fonttype':'none','savefig.facecolor':'white'})
BLUE='#729DBB';RED='#EB7777';AMBER='#D9A441';DARK='#253646'
STATES=['raw','temperature_scaled_v1_1'];ARMS=['RAW','SNV','PREFIX','FULL_TARGET'];TESTS=[2,3,4,5]
ARM_NAMES={'RAW':'Raw spectra','SNV':'SNV','PREFIX':'Initial-window normalization','FULL_TARGET':'Full-target normalization'}
DOMAIN_COLORS={2:'#0072B2',3:'#D55E00',4:'#009E73',5:'#CC79A7'}
STAGE_COLORS={1:'#440154',2:'#3B528B',3:'#21918C',4:'#5EC962',5:'#E6CE33'}
exports=[]
def save(fig,name,caption):
    for ext in ['png','pdf','svg']:
        fig.savefig(FIG/f'{name}.{ext}',dpi=600,bbox_inches='tight',pad_inches=.12)
    exports.append({'figure':name,'caption':caption})
    plt.show();plt.close(fig)
print('NB06:',NB06,'\nNB15:',NB15,'\nOutput:',RUN)


In [ ]:
# Verify frozen inputs. Bin tables reproduce ECE; model identities come from internal selection.
paths={
'bins':NB06/'tables/reliability_bins_all_models_v1_1.csv',
'metrics':NB06/'tables/outer_calibration_metrics_all_models_v1_1.csv',
'selected':NB06/'tables/internally_selected_models_calibration_summary_v1_1.csv'}
h=pd.read_csv(NB06/'metadata/package_hashes.csv');hm=dict(zip(h.relative_path,h.sha256))
for p in paths.values():assert sha(p)==hm[str(p.relative_to(NB06))],p
bins=pd.read_csv(paths['bins']);metrics=pd.read_csv(paths['metrics']);selected=pd.read_csv(paths['selected'])
assert not metrics[['outer_test','modality','model','state']].duplicated().any()
audit=[]
for key,g in bins.groupby(['outer_test','modality','model','state']):
    t,mod,model,state=key;r=metrics[(metrics.outer_test==t)&metrics.modality.eq(mod)&metrics.model.eq(model)&metrics.state.eq(state)]
    assert len(r)==1;r=r.iloc[0]
    e=float(np.sum(g.n*np.abs(g.mean_confidence-g.accuracy))/g.n.sum())
    assert abs(e-r.ece)<1e-10 and g.n.sum()==r.n_test
    assert g.bin.between(1,15).all() and g.n.gt(0).all()
    assert np.all(g.mean_confidence>=(g.bin-1)/15-1e-12) and np.all(g.mean_confidence<=g.bin/15+1e-12)
    audit.append({'outer_test':t,'modality':mod,'model':model,'state':state,'ece_recomputed':e,'ece_saved':r.ece,'abs_difference':abs(e-r.ece),'n':int(g.n.sum())})
pd.DataFrame(audit).to_csv(TABLE/'reliability_numerical_audit.csv',index=False)
z=zipfile.ZipFile(NB15);assert z.testzip() is None
prefix=next(n for n in z.namelist() if n.endswith('/metadata/run_manifest.json')).removesuffix('metadata/run_manifest.json')
def zcsv(name):return pd.read_csv(io.BytesIO(z.read(prefix+name)),compression='gzip' if name.endswith('.gz') else None)
zh=zcsv('metadata/package_hashes.csv')
for r in zh.itertuples():assert hashlib.sha256(z.read(prefix+r.relative_path)).hexdigest()==r.sha256
coords=zcsv('tables/projection_coordinates.csv.gz');var=zcsv('tables/pca_explained_variance.csv');loads=zcsv('tables/pca_coefficients.csv')
print('Verified ECE in',len(audit),'model-state cells; NB15 source hashes:',len(zh))


In [ ]:
# Reliability figures: same scale, actual bin mean as reference, empty bins omitted.
reliability_keys=[]
for mod in ['Spectral','TPH','TPH_Spectral']:
    choices=selected[selected.modality.eq(mod)].sort_values('outer_test')
    assert len(choices)==4
    for state in STATES:
        fig=plt.figure(figsize=(10,9.5));outer=fig.add_gridspec(2,2,wspace=.23,hspace=.35)
        for j,row in enumerate(choices.itertuples()):
            key=(metrics.outer_test.eq(row.outer_test)&metrics.modality.eq(mod)&metrics.model.eq(row.model)&metrics.state.eq(state))
            m=metrics[key].iloc[0]
            g=bins[bins.outer_test.eq(row.outer_test)&bins.modality.eq(mod)&bins.model.eq(row.model)&bins.state.eq(state)].sort_values('bin')
            if mod=='Spectral':assert str(m.probability_source).startswith('native_')
            native=str(m.probability_source).startswith('native_')
            sub=outer[j//2,j%2].subgridspec(2,1,height_ratios=[4,1],hspace=.08)
            ax=fig.add_subplot(sub[0]);hist=fig.add_subplot(sub[1],sharex=ax)
            x=(g.bin.to_numpy()-.5)/15;acc=g.accuracy.to_numpy();conf=g.mean_confidence.to_numpy();gap=conf-acc
            ax.bar(x,acc,width=.9/15,color=BLUE,edgecolor='white',linewidth=.6,zorder=2)
            ax.bar(x,np.abs(gap),bottom=np.minimum(acc,conf),width=.9/15,color=np.where(gap>=0,RED,AMBER),alpha=.85,edgecolor='white',linewidth=.4,zorder=3)
            ax.scatter(x,conf,color=DARK,s=15,zorder=4)
            ax.plot([0,1],[0,1],ls='--',color=DARK,lw=1,zorder=1)
            ax.set(xlim=(0,1),ylim=(0,1),ylabel='Observed accuracy')
            ax.set_yticks([0,.25,.5,.75,1]);ax.tick_params(axis='x',labelbottom=False)
            ax.grid(axis='y',color='#E8ECEF',lw=.6);ax.set_axisbelow(True)
            label='Native probabilities' if native else 'Diagnostic score probabilities'
            title=f'({chr(97+j)}) Test {row.outer_test} | {row.model}\nECE = {m.ece:.3f}   Brier = {m.brier:.3f}   n = {int(m.n_test):,}'
            ax.set_title(title,loc='left',pad=8,fontsize=10.5)
            hist.bar(x,100*g.n/g.n.sum(),width=.9/15,color='#B6C4CD',edgecolor='white',linewidth=.4)
            hist.set(xlim=(0,1),xlabel='Predicted confidence',ylabel='Obs. (%)');hist.set_xticks([0,.25,.5,.75,1]);hist.tick_params(labelsize=8)
            if not native:ax.text(.02,.98,'Diagnostic surrogate',transform=ax.transAxes,va='top',fontsize=8,color='#9A3B00')
            reliability_keys.append({'modality':mod,'outer_test':row.outer_test,'model':row.model,'state':state,'probability_source':m.probability_source,'n':int(m.n_test),'ece':m.ece,'brier':m.brier,'nll':m.nll})
        legend=[Patch(facecolor=BLUE,label='Accuracy'),Patch(facecolor=RED,label='Overconfidence gap'),Patch(facecolor=AMBER,label='Underconfidence gap'),Line2D([0],[0],marker='o',color='none',markerfacecolor=DARK,label='Mean bin confidence'),Line2D([0],[0],ls='--',color=DARK,label='Perfect calibration')]
        fig.legend(handles=legend,loc='lower center',bbox_to_anchor=(.5,-.025),ncol=3,frameon=False,fontsize=9)
        title_state='Before calibration' if state=='raw' else 'After development-fitted temperature scaling'
        fig.suptitle(f'{mod.replace("_"," + ")} | {title_state}',y=.985,fontsize=14,fontweight='semibold')
        name=f'F16_Reliability_{mod}_{"Raw" if state=="raw" else "Calibrated"}'
        save(fig,name,'Top-label reliability with the original 15 equal-width bins. Accuracy, confidence gaps and mean confidence use actual saved bin summaries. Empty bins are omitted. Histograms show observation support. Model identity is selected internally, never from external performance. Temperature is fitted on development only. Population is the original RAW confirmatory analysis, not NB14/NB15.')
pd.DataFrame(reliability_keys).to_csv(TABLE/'figure_reliability_metrics.csv',index=False)


In [ ]:
# PCA and UMAP: categorical legends and clear role markers; no refitting or visual tuning.
for t in TESTS:
    for colorby,palette,title in [('test_id',DOMAIN_COLORS,'Acquisition experiment'),('stage',STAGE_COLORS,'Ripeness stage')]:
        fig,axes=plt.subplots(2,4,figsize=(16,8.4),layout='constrained')
        for j,arm in enumerate(ARMS):
            g=coords[coords.external_test.eq(t)&coords.arm.eq(arm)]
            for i,method in enumerate(['PCA','UMAP']):
                x,y=('PC1','PC2') if method=='PCA' else ('UMAP1','UMAP2');ax=axes[i,j]
                for role,marker,size,alpha in [('development','o',9,.23),('external','^',23,.8)]:
                    h=g[g.role.eq(role)];ax.scatter(h[x],h[y],c=[palette[int(v)] for v in h[colorby]],marker=marker,s=size,alpha=alpha,edgecolors='black' if role=='external' else 'none',linewidths=.25,rasterized=True)
                if method=='PCA':
                    vv=var[var.external_test.eq(t)&var.arm.eq(arm)].set_index('PC').explained_variance_ratio
                    ax.set_xlabel(f'PC1 ({100*vv.loc[1]:.1f}%)');ax.set_ylabel(f'PC2 ({100*vv.loc[2]:.1f}%)')
                else:ax.set_xlabel('UMAP 1');ax.set_ylabel('UMAP 2')
                ax.set_title(f'({chr(97+i*4+j)}) {ARM_NAMES[arm]}',loc='left',fontsize=10)
                ax.tick_params(labelsize=8);ax.grid(alpha=.12)
        handles=[Line2D([0],[0],marker='o',ls='',color=c,label=f'{"Test" if colorby=="test_id" else "Stage"} {k}') for k,c in palette.items()]
        handles+=[Line2D([0],[0],marker='o',ls='',color='#555555',label='Development'),Line2D([0],[0],marker='^',ls='',color='#555555',label=f'Held-out Test {t}')]
        fig.legend(handles=handles,loc='outside lower center',ncol=7,frameon=False,fontsize=9)
        fig.suptitle(f'{title} | external Test {t}',fontsize=14,fontweight='semibold')
        save(fig,f'F16_Projections_T{t}_{colorby}','PCA and UMAP reproduce NB15 coordinates without refitting. Transformations and embeddings fitted on development only. External points are triangles. Separate panels have separate coordinate systems. FULL_TARGET is transductive, and centering mechanically reduces location differences; visual proximity is not predictive validation.')
    fig,axes=plt.subplots(2,2,figsize=(10,7.2),sharex=True,sharey=True,layout='constrained')
    bound=float(np.abs(loads[loads.external_test.eq(t)].coefficient_standardized_space).max())*1.15
    for ax,arm,letter in zip(axes.ravel(),ARMS,'abcd'):
        for k,col,mark in [(1,'#0072B2','o'),(2,'#D55E00','s'),(3,'#009E73','^')]:
            h=loads[loads.external_test.eq(t)&loads.arm.eq(arm)&loads.PC.eq(k)].sort_values('wavelength_nm')
            ev=var[var.external_test.eq(t)&var.arm.eq(arm)&var.PC.eq(k)].explained_variance_ratio.iloc[0]
            ax.plot(h.wavelength_nm,h.coefficient_standardized_space,color=col,marker=mark,ms=3,lw=1.4,label=f'PC{k}: {100*ev:.1f}%')
        ax.axhline(0,color='#888888',ls='--',lw=.7);ax.set_ylim(-bound,bound);ax.set_xticks([410,535,680,810,940]);ax.set_title(f'({letter}) {ARM_NAMES[arm]}',loc='left');ax.legend(frameon=False,fontsize=8,loc='best');ax.grid(alpha=.12)
    fig.supxlabel('Wavelength (nm)');fig.supylabel('PCA coefficient in standardized feature space');fig.suptitle(f'Spectral PCA coefficients | held-out Test {t} excluded from fitting',fontsize=13,fontweight='semibold')
    save(fig,f'F16_Loadings_T{t}','PCA coefficients at 18 discrete channels, without smoothing. Percentages refer to development explained variance. Sign is anchored for display only. Coefficients are not predictive importance or proof of biochemical mechanisms.')


In [ ]:
# Reformat the error summary, retaining source values exactly.
by=zcsv('tables/metrics_by_experiment.csv');loo=zcsv('tables/leave_one_experiment_out_summary.csv');boot=zcsv('tables/conditional_resampling_sensitivity.csv')
fig,axes=plt.subplots(1,3,figsize=(13.4,4.5),layout='constrained')
x=np.arange(4);w=.34
axes[0].bar(x-w/2,by.raw_f1,w,color='#A6ADB2',label='RAW');axes[0].bar(x+w/2,by.prefix_f1,w,color='#0072B2',label='Initial window')
axes[0].set(xticks=x,xticklabels=TESTS,xlabel='Held-out experiment',ylabel='Macro-F1',ylim=(0,.4),title='(a) External classification');axes[0].legend(frameon=False,fontsize=9)
for col,color,label in [('mean_delta_f1','#0072B2','Macro-F1')]:axes[1].plot(loo.omitted_test,loo[col],marker='o',color=color)
axes[1].axhline(0,color=DARK,ls='--',lw=.8);axes[1].set(xticks=TESTS,xlabel='Omitted experiment',ylabel='Mean Macro-F1 change',title='(b) Dependence on experiment')
for offset,method,col,label in [(0,'stage_stratified_rows','#A6ADB2','Stratified rows'),(.18,'signature_clusters','#0072B2','Spectral signatures')]:
    g=boot[boot.method.eq(method)&boot.metric.eq('delta_macro_f1')].sort_values('outer_test');yy=np.arange(4)+offset
    axes[2].hlines(yy,g.q025,g.q975,color=col,lw=2,label=label);axes[2].scatter(g.point,yy,color=col,s=20)
axes[2].set(yticks=np.arange(4)+.09,yticklabels=TESTS,xlabel='Macro-F1 change',ylabel='Held-out experiment',title='(c) Conditional interval sensitivity');axes[2].axvline(0,color=DARK,ls='--',lw=.8);axes[2].legend(frameon=False,fontsize=8)
fig.suptitle('Initial-window normalization | N = 25 unique spectral signatures',fontsize=14,fontweight='semibold')
save(fig,'F16_Error_Robustness_Synthesis','Same NB14 primary observations within each test. Changes are PREFIX minus RAW. Leave-one-experiment-out summaries and conditional resampling are descriptive; signatures are not validated independent biological units. Row and signature resampling also differ in stage-composition constraints. No population significance claim is made.')


In [ ]:
# Provenance and automatic ZIP download.
pd.DataFrame(exports).to_csv(TABLE/'figure_captions.csv',index=False)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'nb06_folder':NB06.name,'nb15_zip':NB15.name,'nb15_sha256':sha(NB15),'source_table_hashes':{k:sha(v) for k,v in paths.items()},'classifier_refits':0,'calibration_refits':0,'embedding_refits':0,'reliability_bins':15,'figures':len(exports),'scope':'Presentation only; primary and exploratory populations kept separate'}
(META/'run_manifest.json').write_text(json.dumps(record,indent=2))
(META/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
(RUN/'READ_ME.md').write_text('Presentation-only figure package. Reliability uses the original RAW NB06-v1.1 population. Projections and error sensitivity use NB15 N=25. Raw and calibrated argmax labels are unchanged by temperature scaling. Histogram support matters; a small ECE alone is not evidence of informative prediction. See figure_captions.csv for panel interpretation.',encoding='utf-8')
h=[]
for p in sorted(RUN.rglob('*')):
    if p.is_file():h.append({'relative_path':str(p.relative_to(RUN)),'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(h).to_csv(META/'package_hashes.csv',index=False)
zip_path=Path(shutil.make_archive(str(RESULTS/('RESULTS_'+RUN.name)),'zip',root_dir=RUN.parent,base_dir=RUN.name))
with zipfile.ZipFile(zip_path) as zz:assert zz.testzip() is None
print('COMPLETADO. Adjunta este ZIP en el chat:',zip_path)
try:files.download(str(zip_path))
except Exception as e:print('Descarga automática no completada; el ZIP está en 06_RESULTS.',repr(e))
